# 03. Acceptance와 End-to-End Speedup

**학습 목표**: decode-only 가속이 vision encode·prefill·layout을 포함하면 왜 줄어드는지 계산하고, acceptance length와 batch 포화의 영향을 탐색합니다. 수치는 블로그 결과를 설명하기 위한 비용 모형이며 benchmark 재현이 아닙니다.

**실행 방법**: Python 3과 Jupyter에서 셀을 위에서 아래로 실행합니다. 외부 패키지는 필요하지 않습니다.

In [ ]:
def end_to_end_speedup(decode_speedup, fixed_fraction):
    # 전체 시간 중 fixed_fraction은 speculation으로 줄지 않습니다.
    return 1 / (fixed_fraction + (1 - fixed_fraction) / decode_speedup)

decode_speedup = 3.85
for fixed in (0.0, 0.2, 0.4, 0.6, 0.8):
    print(f'고정 비용 {fixed:.0%}: {end_to_end_speedup(decode_speedup, fixed):.2f}x')


In [ ]:
def infer_fixed_fraction(decode_speedup, observed_speedup):
    return (1 / observed_speedup - 1 / decode_speedup) / (1 - 1 / decode_speedup)

crop_fixed = infer_fixed_fraction(3.85, 1.85)
page_fixed = infer_fixed_fraction(3.85, 1.34)
print('crop 기준 추정 고정 비용 비중:', f'{crop_fixed:.1%}')
print('page 기준 추정 고정 비용 비중:', f'{page_fixed:.1%}')
assert 0 < crop_fixed < page_fixed < 1


## Acceptance length와 draft step

한 round에 draft `D`회와 verify 1회를 쓴다고 단순화합니다. AR 불일치 token 하나까지 확정하므로 forward당 확정량은 `(acceptance + 1) / (D + 1)`입니다.

In [ ]:
experiments = [
    {'draft_steps': 1, 'accepted': 17.0},
    {'draft_steps': 2, 'accepted': 20.0},
    {'draft_steps': 3, 'accepted': 22.0},
    {'draft_steps': 4, 'accepted': 24.7},
]
for row in experiments:
    row['tokens_per_forward'] = (row['accepted'] + 1) / (row['draft_steps'] + 1)
    print(row)
best = max(experiments, key=lambda row: row['tokens_per_forward'])
assert best['draft_steps'] == 1
print('이 모형의 최적 draft step:', best['draft_steps'])


## Batch 포화

아래 toy 함수는 batch가 커질수록 AR도 GPU를 채워 speculation의 상대 이득이 1.08배 근처로 줄어드는 현상을 표현합니다.

In [ ]:
def batch_speedup(batch, low_batch=1.85, saturated=1.08):
    return saturated + (low_batch - saturated) / batch

for batch in (1, 2, 4, 8, 16, 32):
    print(f'batch={batch:2d}: {batch_speedup(batch):.3f}x')
assert batch_speedup(1) == 1.85
assert batch_speedup(32) < batch_speedup(2)


## 확장 과제

실제 서비스의 output length 분포, vision encode latency, batch arrival rate를 넣어 p50/p95 latency를 계산하세요. 표·수식·본문의 acceptance를 따로 측정하고 품질 하락 비용도 목적함수에 추가해 보세요.